In [41]:
import pandas as pd
import numpy as np
import json

def read_finnish_io_matrix(file_path):
    """
    Read and process the Finnish Input-Output matrix from Excel file
    """
    # Read the Excel file
    df = pd.read_excel(file_path, sheet_name='310_14yn_2022', header=2, usecols="B,D:CC",na_values=["."])
        
    # Remove empty rows and columns
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    
    
    # Keep only rows 0 to 82 (before metadata starts)
    df = df.iloc[:82]
    
    # # Set the first remaining column as index (sector names)
    df = df.set_index(df.columns[0])
    
    # # Clean up column names
    df.columns = [str(col).strip() for col in df.columns]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NAs
    df = df.fillna(0)

    df = make_square(df)
    
    # Convert np.float to regular float
    # sectors = df.index.tolist()
    # for i, source_sector in enumerate(sectors):
    #     for j, target_sector in enumerate(sectors):
    #         df.iloc[i, j] = float(df.iloc[i, j])
            
    return df


def make_square(df):
    """
    Restrict DataFrame to rows/columns that are present in both
    the index and the columns.
    """
    common = df.index.intersection(df.columns)
    return df.loc[common, common]


In [42]:
file_path = "Suomi_pt_2022.xlsx"
io_matrix = read_finnish_io_matrix(file_path)

In [79]:
# Jätetään Kiinteän pääoman muodostus toistaiseksi pois
io_matrix = io_matrix.iloc[:-1, :-1]

io_matrix = io_matrix.iloc[:30, :30]


In [80]:
# Full processing
def process_finnish_io_data(file_path):
    """
    Complete processing of Finnish IO data for visualization
    """
    # Read and process the IO matrix
    io_matrix = read_finnish_io_matrix(file_path)
    
    io_matrix = make_square(io_matrix)
    
    # Prepare network data
    nodes, transactions = prepare_network_data(io_matrix, threshold=0.5)
    
    # Export to JSON
    with open('sectors.json', 'w') as f:
        json.dump(nodes, f, indent=2)
    
    with open('transactions.json', 'w') as f:
        json.dump(transactions, f, indent=2)
    
    # Also save the processed matrix
    io_matrix.to_csv('processed_io_matrix.csv')
    
    return io_matrix, nodes, transactions

In [81]:
def read_finnish_emission_matrix(file_path):
    """
    Read and process the Finnish Input-Output matrix from Excel file
    """
    # Read the Excel file
    df = pd.read_excel(file_path, sheet_name='001_11ig_2022', header=2, usecols="B:Q",na_values=["."])
        
    # Remove empty rows and columns
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    
    # Keep only rows 0 to 82 (before metadata starts)
    df = df.iloc[:93]
    
    # # Set the first remaining column as index (sector names)
    df = df.set_index(df.columns[0])
    
    # # Clean up column names
    df.columns = [str(col).strip() for col in df.columns]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NAs
    df = df.fillna(0)
    
    return df

In [82]:
file_path_emissions = "Suomi_päästöt_2022.xlsx"
emis_df = read_finnish_emission_matrix(file_path_emissions)

In [83]:
#len(emis_df.index.intersection(io_matrix.columns)) #54
#len(io_matrix.index) #64

In [84]:
set(emis_df.index).difference(set(emis_df.index.intersection(io_matrix.index).tolist()))
# Eivät mene yksi yhteen, vaan vaatii hieman manuaalista väsäämistä

{'13-15 Tekstiili-, vaatetus- ja nahkateollisuus',
 '16-17 Metsäteollisuus',
 '19-22 Kemianteollisuus',
 '26-27 Sähkö- ja elektroniikkateollisuus',
 '31-32 Muu valmistus ml. huonekalut',
 '49 Maaliikenne ja putkijohtokuljetus',
 '50 Vesiliikenne',
 '51 Ilmaliikenne',
 '52 Varastointi ja liikennettä palveleva toiminta',
 '53 Posti- ja kuriiritoiminta',
 '58 Kustannustoiminta',
 '59-60 Audiovisuaalinen toiminta',
 '61 Televiestintä',
 '62-63 Tietojenkäsittelypalvelu',
 '64 Rahoituspalvelut (pl. vakuutus- ja eläkevakuutustoiminta)',
 '65 Vakuutus-, jälleenvakuutus- ja eläkevakuutustoiminta (pl. pakollinen sosiaalivakuutus)',
 '66 Rahoitusta ja vakuuttamista palveleva toiminta',
 '68 Kiinteistöalan toiminta',
 '69-70 Liikkeenjohdon palvelut',
 '71 Arkkitehti- ja insinööripalvelut; tekninen testaus ja analysointi',
 '72 Tieteellinen tutkimus ja kehittäminen',
 '73 Mainostoiminta ja markkinatutkimus',
 '74-75 Muut liike-elämän palvelut ja eläinlääkintä',
 '77 Vuokraus- ja leasingtoiminta',
 

In [85]:
päästöt = emis_df.loc[emis_df.index.intersection(io_matrix.columns), :]
päästöt = päästöt.iloc[:,-1]

In [86]:
def prepare_network_data(io_matrix, emission_vector, threshold=0.1):
    """
    Prepare IO matrix data for network visualization
    
    Parameters:
    io_matrix: Processed IO matrix DataFrame
    threshold: Minimum flow value to include (as percentage of max flow)
    """
    # Get sector names
    sectors = io_matrix.index.tolist()
    
    # Create a list of transactions (edges)
    transactions = []
    
    # Calculate threshold value
    max_flow = io_matrix.values.max()
    threshold_value = max_flow * threshold / 100
    
    # Iterate through the matrix to create edges
    for i, source_sector in enumerate(sectors):
        for j, target_sector in enumerate(sectors):
            flow_value = io_matrix.iloc[i, j]
            
            # Only include flows above the threshold
            if True: #flow_value > threshold_value:
                transactions.append({
                    'source': source_sector,
                    'target': target_sector,
                    'value': flow_value
                })

    all_emissions = sum(emission_vector)
    
    # Create nodes with additional information
    nodes = []
    for sector in sectors:
        # Calculate total output and input for each sector
        total_output = io_matrix.loc[sector].sum()
        total_input = io_matrix[sector].sum()

        try:
            emissions = float(emission_vector.loc[sector])
            relative_emissions = emissions/all_emissions * 100 # prosentteina
            value_created_per_emissions = total_output / emissions  *1000000
            
        except Exception as e:
            emissions = 0
            relative_emissions = 0
            value_created_per_emissions = 0

        # if value_created_per_emissions > 10^20:
        #     value_created_per_emissions = 0
        
        
        # Categorize sectors (simplified categorization)
        # TODO muuta suomenkieliseksi, WIP
        if any(keyword in sector for keyword in ['viljely', 'Viljely', 'Kalastus', 'Metsä', 'metsä']):
            group = 'primary'
        elif any(keyword in sector for keyword in ['Teollisuus', 'teollisuus', 'valmistus']):
            group = 'manufacturing'
        elif any(keyword in sector for keyword in ['Energy', 'Electric', 'Gas']):
            group = 'energy'
        elif any(keyword in sector for keyword in ['kauppa', 'Kauppa', 'Huolto', 'Jakelu','jakelu', 'koulutus', 'palvelu', 'Palvelu', 'toiminta', 'Toiminta','Kuljetus']):
            group = 'service'
        else:
            group = 'service'
        
        nodes.append({
            'id': sector,
            'name': sector,
            'group': group,
            'value': total_output, # Using total output as node size indicator
            'emissions' : emissions,
            'relative_emissions': relative_emissions,
            'value_created_per_emissions' : value_created_per_emissions
        })
    
    return nodes, transactions

In [87]:
nodes, transactions = prepare_network_data(io_matrix, päästöt, threshold=10)

/tmp/ipykernel_8646/4268802432.py:44: RuntimeWarning: divide by zero encountered in scalar divide
  value_created_per_emissions = total_output / emissions  *1000000


In [88]:
# Export nodes and transactions to JSON files
with open('sectors.json', 'w') as f:
    json.dump(nodes, f, indent=2)

with open('transactions.json', 'w') as f:
    json.dump(transactions, f, indent=2)

print("Data exported to sectors.json and transactions.json")

Data exported to sectors.json and transactions.json


In [75]:
io_matrix.index

Index(['01 Kasvinviljely ja kotieläintalous, riistatalous ja niihin liittyvät palvelut',
       '02 Metsätalous ja puunkorjuu', '03 Kalastus ja vesiviljely',
       'B Kaivostoiminta ja louhinta (05-09)',
       '10-12 Elintarviketeollisuus ym.', '13-15 Tekstiiliteollisuus',
       '16 Sahatavaran sekä puu- ja korkkituotteiden valmistus (pl. huonekalut); olki- ja punontatuotteiden valmistus',
       '17 Paperin, paperi- ja kartonkituotteiden valmistus',
       '18 Painaminen ja tallenteiden jäljentäminen',
       '19 Koksin ja jalostettujen öljytuotteiden valmistus',
       '20 Kemikaalien ja kemiallisten tuotteiden valmistus',
       '21 Lääkeaineiden ja lääkkeiden valmistus',
       '22 Kumi- ja muovituotteiden valmistus',
       '23 Muiden ei-metallisten mineraalituotteiden valmistus',
       '24 Metallien jalostus',
       '25 Metallituotteiden valmistus (pl. koneet ja laitteet)',
       '26 Tietokoneiden sekä elektronisten ja optisten tuotteiden valmistus',
       '27 Sähkölaittei